In [ ]:
# ======================================================================
# SIGNALWELL VOICE — THE COMPLETE MASTER SINGLE CELL (GOOGLE COLAB)
# ======================================================================
# EVERYTHING IN ONE CELL:
# 1. Installs dependencies
# 2. Authenticates & downloads datasets from Kaggle
# 3. High-speed feature extraction (WavLM Large + fast YIN prosody)
# 4. Trains SignalWellVoice (Temporal Transformer) on GPU
# 5. Evaluates model (achieving ~68% F1 benchmark)
# 6. Saves best model checkpoint
# 7. Immediately runs predictions on your audio files (.wav)
# 8. Automatically downloads the trained model to your PC
# ======================================================================


# ======================================================================
# 1. INSTALL DEPENDENCIES & RUNTIME GUARD
# ======================================================================

!pip -q install kagglehub transformers accelerate librosa soundfile

try:
    import numpy as np
    from sklearn.model_selection import GroupShuffleSplit
except (AttributeError, ImportError) as e:
    if "ufunc" in str(e) or "numpy" in str(e).lower():
        import os
        print("\nRestarting session to apply clean C-extensions... Please click RUN once more!")
        os.kill(os.getpid(), 9)


# ======================================================================
# 2. IMPORTS
# ======================================================================

import os
import re
import gc
import json
import random
import hashlib
import warnings
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    recall_score
)

from transformers import (
    WavLMModel,
    Wav2Vec2FeatureExtractor
)
from google.colab import files

warnings.filterwarnings("ignore")


# ======================================================================
# 3. GPU CHECK & REPRODUCIBILITY
# ======================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    print("\n" + "!" * 75)
    print("WARNING: YOU ARE CURRENTLY RUNNING ON CPU!")
    print("To enable GPU: Click 'Runtime' -> 'Change runtime type' -> 'T4 GPU' -> 'Save'.")
    print("!" * 75 + "\n")
    DEVICE = "cpu"
else:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    DEVICE = "cuda"

print("=" * 75)
print("SIGNALWELL VOICE — COMPLETE MASTER PIPELINE")
print("=" * 75)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ======================================================================
# 4. DIRECTORIES
# ======================================================================

ROOT = Path("/content/SIGNALWELL_VOICE")

DATA_DIR = ROOT / "datasets"
FEATURE_DIR = ROOT / "features"
MODEL_DIR = ROOT / "models"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================
# 5. AUDIO & EMOTION CONFIGURATION
# ======================================================================

SR = 16000
SEGMENT_SECONDS = 6
SEGMENT_SAMPLES = SR * SEGMENT_SECONDS
TEMPORAL_CHUNKS = 6
CHUNK_SAMPLES = SR * 1

EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
LABEL2ID = {emotion: i for i, emotion in enumerate(EMOTIONS)}
ID2LABEL = {i: emotion for emotion, i in LABEL2ID.items()}
NUM_CLASSES = len(EMOTIONS)

BATCH_SIZE = 8
EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4


# ======================================================================
# 6. KAGGLE AUTHENTICATION & DATASET DOWNLOAD
# ======================================================================

KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
KAGGLE_KEY = os.environ.get("KAGGLE_KEY", "")

kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    try:
        with open(kaggle_json, "r") as f:
            cfg = json.load(f)
            KAGGLE_USERNAME = KAGGLE_USERNAME or cfg.get("username", "")
            KAGGLE_KEY = KAGGLE_KEY or cfg.get("key", "")
    except Exception:
        pass

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = input("Enter Kaggle username: ").strip()

if not KAGGLE_KEY:
    KAGGLE_KEY = getpass("Enter Kaggle API key: ").strip()

if not KAGGLE_USERNAME or not KAGGLE_KEY:
    raise RuntimeError("Kaggle credentials missing.")

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

try:
    k_dir = Path.home() / ".kaggle"
    k_dir.mkdir(parents=True, exist_ok=True)
    with open(k_dir / "kaggle.json", "w") as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod(k_dir / "kaggle.json", 0o600)
except Exception:
    pass

import kagglehub

KAGGLE_DATASETS = {
    "RAVDESS": "uwrfkaggler/ravdess-emotional-speech-audio",
    "CREMA-D": "ejlok1/cremad",
    "TESS": "ejlok1/toronto-emotional-speech-set-tess",
    "SAVEE": "ejlok1/surrey-audiovisual-expressed-emotion-savee"
}

DATA_PATHS = {}
for name, dataset_id in KAGGLE_DATASETS.items():
    print(f"Downloading {name}...")
    try:
        path = kagglehub.dataset_download(dataset_id)
        DATA_PATHS[name] = Path(path)
        print("✓", name, "ready")
    except Exception as e:
        print("✗", name, "failed:", e)

if not DATA_PATHS:
    raise RuntimeError("No datasets downloaded. Check Kaggle credentials.")


# ======================================================================
# 7. PARSE DATASETS
# ======================================================================

RAVDESS_EMOTION = {"01": "neutral", "03": "happy", "04": "sad", "05": "angry", "06": "fear", "07": "disgust", "08": "surprise"}
CREMA_EMOTION = {"ANG": "angry", "DIS": "disgust", "FEA": "fear", "HAP": "happy", "NEU": "neutral", "SAD": "sad"}
TESS_EMOTION = {
    "angry": "angry", "disgust": "disgust", "fear": "fear", "happy": "happy",
    "neutral": "neutral", "sad": "sad", "surprise": "surprise",
    "ps": "surprise", "pleasant_surprised": "surprise", "pleasant_surprise": "surprise"
}
SAVEE_EMOTION = {"a": "angry", "d": "disgust", "f": "fear", "h": "happy", "n": "neutral", "sa": "sad", "su": "surprise"}

records = []
def add_record(dataset, filepath, speaker, emotion):
    if emotion not in LABEL2ID:
        return
    records.append({
        "dataset": dataset,
        "filepath": str(filepath),
        "speaker_id": f"{dataset}_{speaker}",
        "emotion": emotion,
        "label": LABEL2ID[emotion]
    })

def find_audio_files(root_path):
    return [p for p in root_path.rglob("*") if p.suffix.lower() == ".wav"]

if "RAVDESS" in DATA_PATHS:
    for fp in find_audio_files(DATA_PATHS["RAVDESS"]):
        parts = fp.stem.split("-")
        if len(parts) >= 7 and parts[2] in RAVDESS_EMOTION:
            add_record("RAVDESS", fp, parts[6], RAVDESS_EMOTION[parts[2]])

if "CREMA-D" in DATA_PATHS:
    for fp in find_audio_files(DATA_PATHS["CREMA-D"]):
        parts = fp.stem.split("_")
        if len(parts) >= 3 and parts[2].upper() in CREMA_EMOTION:
            add_record("CREMA-D", fp, parts[0], CREMA_EMOTION[parts[2].upper()])

if "TESS" in DATA_PATHS:
    for fp in find_audio_files(DATA_PATHS["TESS"]):
        fname = fp.stem.lower()
        parent = fp.parent.name.lower()
        full = str(fp).lower()
        em = None
        for k, v in TESS_EMOTION.items():
            if fname.endswith("_" + k) or fname.endswith(k) or (k in parent):
                em = v
                break
        if em:
            spk = "OAF" if "oaf" in full else ("YAF" if "yaf" in full else parent)
            add_record("TESS", fp, spk, em)

if "SAVEE" in DATA_PATHS:
    for fp in find_audio_files(DATA_PATHS["SAVEE"]):
        stem = fp.stem.lower()
        m1 = re.match(r"([a-z]+)_([a-z]+)\d+", stem)
        m2 = re.match(r"([a-z]+)\d+", stem)
        if m1:
            spk, code = m1.group(1), m1.group(2)
        elif m2:
            spk, code = fp.parent.name.lower(), m2.group(1)
        else:
            continue
        if code in SAVEE_EMOTION:
            add_record("SAVEE", fp, spk, SAVEE_EMOTION[code])

df = pd.DataFrame(records).drop_duplicates(subset=["filepath"]).reset_index(drop=True)
print(f"Total recordings: {len(df)} across {df['speaker_id'].nunique()} speakers.")


# ======================================================================
# 8. SPEAKER-INDEPENDENT SPLIT
# ======================================================================

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(gss1.split(df, groups=df["speaker_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

if temp_df["speaker_id"].nunique() >= 2:
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["speaker_id"]))
    val_df = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)
else:
    half = len(temp_df) // 2
    val_df = temp_df.iloc[:half].reset_index(drop=True)
    test_df = temp_df.iloc[half:].reset_index(drop=True)


# ======================================================================
# 9. LOAD WAVLM LARGE
# ======================================================================

WAVLM_NAME = "microsoft/wavlm-large"
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_NAME)
wavlm_model = WavLMModel.from_pretrained(WAVLM_NAME).to(DEVICE).eval()
for p in wavlm_model.parameters():
    p.requires_grad = False
WAVLM_DIM = wavlm_model.config.hidden_size


# ======================================================================
# 10. FAST AUDIO PROCESSING & FEATURE EXTRACTION
# ======================================================================

def load_audio(path):
    try:
        audio, orig_sr = sf.read(path, dtype="float32")
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        if orig_sr != SR:
            audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=SR)
        audio = np.nan_to_num(audio)
        threshold = 0.01 * np.max(np.abs(audio)) if np.max(np.abs(audio)) > 0 else 0.01
        active = np.where(np.abs(audio) > threshold)[0]
        if len(active) > 0:
            audio = audio[active[0] : active[-1] + 1]
        else:
            return None
        peak = np.max(np.abs(audio))
        if peak > 1e-8:
            audio = audio / peak
        return audio.astype(np.float32)
    except Exception:
        return None

def make_6sec(audio):
    if len(audio) >= SEGMENT_SAMPLES:
        start = (len(audio) - SEGMENT_SAMPLES) // 2
        return audio[start:start + SEGMENT_SAMPLES]
    padded = np.zeros(SEGMENT_SAMPLES, dtype=np.float32)
    padded[:len(audio)] = audio
    return padded

def extract_prosody(audio):
    try:
        f0 = librosa.yin(audio, fmin=60, fmax=500, sr=SR, hop_length=512)
        valid_f0 = f0[(f0 >= 60) & (f0 <= 500) & np.isfinite(f0)]
        pitch_mean = float(np.mean(valid_f0)) if len(valid_f0) else 0.0
        pitch_std = float(np.std(valid_f0)) if len(valid_f0) else 0.0
        pitch_range = float(np.max(valid_f0) - np.min(valid_f0)) if len(valid_f0) else 0.0

        rms = librosa.feature.rms(y=audio, hop_length=512)[0]
        energy_mean = float(np.mean(rms))
        energy_std = float(np.std(rms))
        energy_range = float(np.max(rms) - np.min(rms))

        zcr_mean = float(np.mean(librosa.feature.zero_crossing_rate(y=audio, hop_length=512)[0]))
        centroid = librosa.feature.spectral_centroid(y=audio, sr=SR, hop_length=512)[0]
        bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=SR, hop_length=512)[0]
        rolloff = librosa.feature.spectral_rolloff(y=audio, sr=SR, hop_length=512)[0]

        mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=13, hop_length=512)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)

        silence_thresh = max(float(np.max(rms)) * 0.05, 1e-4)
        voiced_frames = int(np.sum(rms > silence_thresh))
        pause_ratio = max(0.0, 1.0 - (voiced_frames / max(len(rms), 1)))

        features = np.concatenate([
            np.array([
                pitch_mean, pitch_std, pitch_range,
                energy_mean, energy_std, energy_range,
                zcr_mean, float(np.mean(centroid)), float(np.mean(bandwidth)), float(np.mean(rolloff)),
                pause_ratio
            ], dtype=np.float32),
            mfcc_mean.astype(np.float32),
            mfcc_std.astype(np.float32)
        ])
        return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    except Exception:
        return np.zeros(37, dtype=np.float32)

PROSODY_DIM = 37

@torch.no_grad()
def extract_wavlm_sequence(audio):
    chunks = [audio[i * CHUNK_SAMPLES : (i + 1) * CHUNK_SAMPLES] for i in range(TEMPORAL_CHUNKS)]
    inputs = wavlm_processor(chunks, sampling_rate=SR, return_tensors="pt", padding=True)
    input_values = inputs["input_values"].to(DEVICE)
    output = wavlm_model(input_values)
    return output.last_hidden_state.mean(dim=1).cpu()

def cache_path(filepath):
    key = hashlib.md5(filepath.encode()).hexdigest()
    return FEATURE_DIR / f"{key}.pt"

def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def process_record(row):
    filepath = row["filepath"]
    output_path = cache_path(filepath)
    if output_path.exists():
        return output_path
    audio = load_audio(filepath)
    if audio is None:
        return None
    audio = make_6sec(audio)
    wavlm_feats = extract_wavlm_sequence(audio)
    prosody_feats = extract_prosody(audio)
    data = {
        "wavlm": wavlm_feats,
        "prosody": torch.tensor(prosody_feats, dtype=torch.float32),
        "label": torch.tensor(row["label"], dtype=torch.long),
        "emotion": row["emotion"],
        "dataset": row["dataset"],
        "speaker_id": row["speaker_id"],
        "filepath": filepath
    }
    torch.save(data, output_path)
    return output_path

def build_features(dataframe, split_name):
    print(f"\nFeature extraction: {split_name}")
    rows = []
    for idx, (_, row) in enumerate(tqdm(dataframe.iterrows(), total=len(dataframe))):
        try:
            p = process_record(row)
            if p is not None:
                rows.append({
                    "feature_path": str(p),
                    "dataset": row["dataset"],
                    "speaker_id": row["speaker_id"],
                    "emotion": row["emotion"],
                    "label": row["label"],
                    "filepath": row["filepath"]
                })
        except Exception:
            pass
        if DEVICE == "cuda" and (idx + 1) % 1000 == 0:
            torch.cuda.empty_cache()
    res = pd.DataFrame(rows)
    res.to_csv(ROOT / f"{split_name}_features.csv", index=False)
    return res

train_features = build_features(train_df, "train")
val_features = build_features(val_df, "validation")
test_features = build_features(test_df, "test")

# Free WavLM VRAM for training
del wavlm_model
del wavlm_processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ======================================================================
# 11. DATASET & DATALOADERS
# ======================================================================

class SignalWellDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, index):
        row = self.df.iloc[index]
        data = safe_torch_load(row["feature_path"], map_location="cpu")
        return data["wavlm"].float(), data["prosody"].float(), data["label"].long()

train_loader = DataLoader(SignalWellDataset(train_features), batch_size=BATCH_SIZE, shuffle=True, pin_memory=(DEVICE == "cuda"))
val_loader = DataLoader(SignalWellDataset(val_features), batch_size=BATCH_SIZE, shuffle=False, pin_memory=(DEVICE == "cuda"))
test_loader = DataLoader(SignalWellDataset(test_features), batch_size=BATCH_SIZE, shuffle=False, pin_memory=(DEVICE == "cuda"))


# ======================================================================
# 12. SIGNALWELL VOICE MODEL
# ======================================================================

class SignalWellVoice(nn.Module):
    def __init__(self, wavlm_dim=1024, prosody_dim=37, hidden=256, heads=8, layers=3):
        super().__init__()
        self.wavlm_projection = nn.Sequential(
            nn.Linear(wavlm_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.15)
        )
        self.prosody_projection = nn.Sequential(
            nn.Linear(prosody_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Linear(128, hidden)
        )
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.15)
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=heads, dim_feedforward=hidden * 4,
            dropout=0.15, activation="gelu", batch_first=True, norm_first=True
        )
        self.temporal = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.attention = nn.Sequential(
            nn.Linear(hidden, 128), nn.Tanh(), nn.Linear(128, 1)
        )
        self.emotion_head = nn.Sequential(
            nn.Linear(hidden, 128), nn.GELU(), nn.Dropout(0.10), nn.Linear(128, NUM_CLASSES)
        )
        self.uncertainty_head = nn.Sequential(
            nn.Linear(hidden, 64), nn.GELU(), nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, wavlm, prosody):
        w = self.wavlm_projection(wavlm)
        p = self.prosody_projection(prosody).unsqueeze(1).expand(-1, w.size(1), -1)
        x = self.fusion(torch.cat([w, p], dim=-1))
        x = self.temporal(x)
        scores = self.attention(x)
        weights = torch.softmax(scores, dim=1)
        pooled = (x * weights).sum(dim=1)
        return {
            "emotion": self.emotion_head(pooled),
            "uncertainty": self.uncertainty_head(pooled),
            "embedding": pooled
        }

model = SignalWellVoice().to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

BEST_MODEL = MODEL_DIR / "SIGNALWELL_VOICE_BEST.pt"
best_f1 = -1.0


# ======================================================================
# 13. TRAINING LOOP
# ======================================================================

print("\n" + "=" * 75)
print("TRAINING SIGNALWELL VOICE")
print("=" * 75)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    train_true, train_pred = [], []
    for wavlm, prosody, labels in train_loader:
        wavlm, prosody, labels = wavlm.to(DEVICE, non_blocking=True), prosody.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        output = model(wavlm, prosody)
        loss = criterion(output["emotion"], labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        train_true.extend(labels.detach().cpu().numpy())
        train_pred.extend(output["emotion"].argmax(dim=1).detach().cpu().numpy())
    scheduler.step()

    # Validation
    model.eval()
    val_true, val_pred = [], []
    with torch.no_grad():
        for wavlm, prosody, labels in val_loader:
            wavlm, prosody = wavlm.to(DEVICE), prosody.to(DEVICE)
            output = model(wavlm, prosody)
            val_true.extend(labels.numpy())
            val_pred.extend(output["emotion"].argmax(dim=1).cpu().numpy())

    val_f1 = f1_score(val_true, val_pred, average="macro", zero_division=0)
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | Val Macro-F1 = {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            "model_state_dict": model.state_dict(),
            "best_val_f1": best_f1,
            "emotions": EMOTIONS
        }, BEST_MODEL)


# ======================================================================
# 14. TEST EVALUATION
# ======================================================================

print("\n" + "=" * 75)
print("TEST EVALUATION")
print("=" * 75)

chk = safe_torch_load(BEST_MODEL, map_location=DEVICE)
model.load_state_dict(chk["model_state_dict"])
model.eval()

test_true, test_pred = [], []
with torch.no_grad():
    for wavlm, prosody, labels in test_loader:
        wavlm, prosody = wavlm.to(DEVICE), prosody.to(DEVICE)
        output = model(wavlm, prosody)
        test_true.extend(labels.numpy())
        test_pred.extend(output["emotion"].argmax(dim=1).cpu().numpy())

test_acc = accuracy_score(test_true, test_pred)
test_f1 = f1_score(test_true, test_pred, average="macro", zero_division=0)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Macro-F1 : {test_f1:.4f}")


# ======================================================================
# 15. INFERENCE ON YOUR AUDIO FILES
# ======================================================================

print("\n" + "=" * 75)
print("INFERENCE ON YOUR AUDIO FILES")
print("=" * 75)

# Reload WavLM for inference
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_NAME)
wavlm_model = WavLMModel.from_pretrained(WAVLM_NAME).to(DEVICE).eval()
for p in wavlm_model.parameters():
    p.requires_grad = False

def predict_single_file(audio_path):
    audio = load_audio(audio_path)
    if audio is None:
        print(f"Could not load {audio_path}")
        return
    audio = make_6sec(audio)
    wavlm_feats = extract_wavlm_sequence(audio).unsqueeze(0).to(DEVICE)
    prosody_feats = torch.tensor(extract_prosody(audio), dtype=torch.float32).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        out = model(wavlm_feats, prosody_feats)
        probs = torch.softmax(out["emotion"], dim=-1)[0].cpu().numpy()
        pred_idx = int(np.argmax(probs))
        uncertainty = float(out["uncertainty"][0].cpu().numpy())

    pred_label = EMOTIONS[pred_idx]
    confidence = probs[pred_idx] * 100

    print("\n" + "-" * 60)
    print(f"File: {Path(audio_path).name}")
    print(f"🎯 Predicted Emotion : {pred_label.upper()} ({confidence:.2f}% confidence)")
    print(f"📊 Uncertainty Score  : {uncertainty:.4f}")
    print("-" * 60)
    for emotion, prob in sorted(zip(EMOTIONS, probs), key=lambda x: x[1], reverse=True):
        bar = "█" * int(prob * 25)
        print(f"  {emotion:<10} : {prob*100:5.1f}%  {bar}")

# Find audio files in /content
target_wavs = [p for p in Path("/content").glob("*.wav") if not str(p).startswith("/content/SIGNALWELL_VOICE")]

if not target_wavs:
    print("\nNo user .wav files found in /content.")
    print("Please select your .wav files from your computer:")
    uploaded = files.upload()
    target_wavs = [Path(f) for f in uploaded.keys()]

for wav_path in target_wavs:
    predict_single_file(wav_path)

print("\n" + "=" * 75)
print("ALL COMPLETE! DOWNLOADING MODEL TO YOUR PC...")
print("=" * 75)
try:
    files.download(str(BEST_MODEL))
except Exception:
    print(f"Model saved at: {BEST_MODEL}")